### Imports

In [ ]:
from helpers import load_csv_dataset
from pathlib import Path

data_dir = Path('../data')
quotations_dataset = load_csv_dataset(data_dir.joinpath('quotations.csv'), delimiter = ';')
respondents_dataset = load_csv_dataset(data_dir.joinpath('respondents.csv'), delimiter = ';')
requirements_dataset = load_csv_dataset(data_dir.joinpath('requirements.csv'))

### Requirements

#### 1. Number of requirements per category

In [ ]:
categories = set([x['category'] for x in requirements_dataset])

for c in categories:
    count = 0
    for req in requirements_dataset:
        count = count + 1 if req['category'] == c else count + 0
    print(f'{c} -> {count}')

#### 2. Relating respondents and requirements

In [ ]:
requirements_and_respondents_dataset = []
for r in requirements_dataset:
    respondents_per_code = set([x['document'] for x in quotations_dataset if r['code'] in x['codes']])
    for respondent in respondents_per_code:
        id = [r1['role'] for r1 in respondents_dataset if r1['id'] == respondent][0]
        requirements_and_respondents_dataset.append({'scenario': r['scenario'], 
                        'category': r['category'], 
                        'complete-requirement': r['complete-requirement'], 
                        'code': r['code'],
                        'respondent-id':respondent,
                        'respondent-role': [r1['role'] for r1 in respondents_dataset if r1['id'] == respondent][0],
                        'respondent-experience':[r1['experience'] for r1 in respondents_dataset if r1['id'] == respondent][0],
                        'respondent-education':[r1['education'] for r1 in respondents_dataset if r1['id'] == respondent][0]})

In [ ]:
requirements_dataset = load_csv_dataset("../data/requirements.csv")
respondents_dataset = load_csv_dataset("../data/respondents.csv", delimiter=';')
requirements_categories = list(set([x['category'] for x in requirements_dataset]))
dataset = []
for respondent in respondents_dataset:
    categories = []
    for r in requirements_dataset:
        respondents_per_code = list(set(extract_respondents_id(r['code'])))
        if respondent['id'] in respondents_per_code:
            categories.append(r['category'])
    row = {'respondent-id': respondent['id'],
           'respondent-role': respondent['role'],
           'respondent-experience': respondent['experience'],
           'respondent-education': respondent['education']}
    
    for cat in requirements_categories:
        if cat in categories:
            row[cat] = 1
        else:
            row[cat] = 0
            
    dataset.append(row)
save_csv_dataset('../data/respondents-per-requirement-category.csv', dataset)

#### 3. Checking most cited requirements

In [ ]:
for r in requirements_dataset:
    count = 0
    for req in requirements_and_respondents_dataset:
        if r['complete-requirement'] == req['complete-requirement']:
            count += 1
    r['n-respondents'] = count

In [ ]:
top_10_most_cited = sorted(requirements_dataset, key=lambda d: d['n-respondents'], reverse=True)[:10]
table_data = ''
for req in top_10_most_cited:
    row = ''
    row += req['complete-requirement'] + ' & ' + req['category'] + ' & ' + str(req['n-respondents']) + '\\\\\n'
    table_data += row
    
with open(data_dir.joinpath("table_templates").joinpath("template-top-10-most-cited-requirements.txt"),'r') as f:
    lines = f.readlines()
    id = [x for x, y in enumerate(lines) if y == '\\data \n']
    lines[id[0]] = table_data
    with open(data_dir.joinpath("table_templates").joinpath("top-10-most-cited-requirements.txt"),'w') as f1:
        f1.writelines(lines)
        f1.close()

#### 4. Checking how many requiremnts were indicated by a multiple respondents or a single respondent

In [ ]:
len([x for x in requirements_dataset if x['n-respondents'] == 1])

In [ ]:
len([x for x in requirements_dataset if x['n-respondents'] > 1])

In [ ]:
len(requirements_dataset)